In [ ]:
# CONFIGURATION - Change this to your log directory
LOG_DIR = '/tmp/rs10-2mb'  # <-- CHANGE THIS TO YOUR LOG DIRECTORY

# After changing LOG_DIR, you can run all cells: Cell -> Run All

# Latency Analysis for Monad BFT

This notebook analyzes latency data from simnet experiments.

In [226]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import re
from pathlib import Path
import glob
import os

# Verify the directory exists
if not os.path.exists(LOG_DIR):
    print(f"ERROR: Directory {LOG_DIR} does not exist!")
    print("Please update LOG_DIR in the first cell to point to your log directory.")
else:
    print(f"Using log directory: {LOG_DIR}")
    stdout_files = glob.glob(f'{LOG_DIR}/*/stdout')
    print(f"Found {len(stdout_files)} node log files")

Using log directory: /tmp/rs10-2mb-plain
Found 10 node log files


In [227]:
def parse_logs(log_dir):
    """Parse logs from the specified directory and extract latency data."""
    
    latencies = []
    message_sizes = []
    nodes = []
    
    # Remove ANSI color codes regex
    ansi_escape = re.compile(r'\x1b\[[0-9;]*m')
    
    # Pattern to extract latency and message size
    pattern = re.compile(r'message received.*latency_ms=([0-9.]+).*message_size=([0-9]+)')
    
    # Read all stdout files
    for stdout_file in glob.glob(f'{log_dir}/*/stdout'):
        node_name = Path(stdout_file).parent.name
        
        with open(stdout_file, 'r') as f:
            for line in f:
                # Remove ANSI codes
                clean_line = ansi_escape.sub('', line)
                
                # Extract latency and message size
                match = pattern.search(clean_line)
                if match:
                    latency = float(match.group(1))
                    message_size = int(match.group(2)) / 1_000_000  # Convert to MB
                    
                    latencies.append(latency)
                    message_sizes.append(message_size)
                    nodes.append(node_name)
    
    # Create DataFrame
    df = pd.DataFrame({
        'latency_ms': latencies,
        'message_size_mb': message_sizes,
        'node': nodes
    })
    
    return df

In [228]:
# Load data using the LOG_DIR variable from the top of the notebook
df = parse_logs(LOG_DIR)

print(f"Loaded {len(df)} data points from {LOG_DIR}")
print(f"\nData preview:")
df.head()

Loaded 1880 data points from /tmp/rs10-2mb-plain

Data preview:


,latency_ms,message_size_mb,node
0,191.547246,2.0,g10
1,197.780268,2.0,g10
2,193.176183,2.0,g10
3,200.504162,2.0,g10
4,207.838189,2.0,g10


In [229]:
# Calculate statistics
stats = {
    'count': len(df),
    'mean': df['latency_ms'].mean(),
    'std': df['latency_ms'].std(),
    'min': df['latency_ms'].min(),
    'max': df['latency_ms'].max(),
    'p50': df['latency_ms'].quantile(0.50),
    'p90': df['latency_ms'].quantile(0.90),
    'p95': df['latency_ms'].quantile(0.95),
    'p99': df['latency_ms'].quantile(0.99),
    'p999': df['latency_ms'].quantile(0.999),
}

print("Latency Statistics:")
print("="*50)
for key, value in stats.items():
    if key == 'count':
        print(f"{key:10s}: {value:,}")
    else:
        print(f"{key:10s}: {value:.2f} ms")

Latency Statistics:
count     : 1,880
mean      : 159.57 ms
std       : 51.98 ms
min       : 12.32 ms
max       : 252.03 ms
p50       : 169.51 ms
p90       : 199.88 ms
p95       : 204.86 ms
p99       : 213.34 ms
p999      : 232.21 ms


In [230]:
# Create interactive histogram with Plotly
fig_hist = go.Figure()

# Add histogram
fig_hist.add_trace(go.Histogram(
    x=df['latency_ms'],
    nbinsx=50,
    name='Latency Distribution',
    marker_color='#2E86AB',
    opacity=0.75
))

# Add vertical lines for percentiles
percentiles = {
    'Mean': (stats['mean'], '#E74C3C'),
    'P50': (stats['p50'], '#F39C12'),
    'P95': (stats['p95'], '#3498DB'),
    'P99': (stats['p99'], '#2ECC71'),
    'P99.9': (stats['p999'], '#9B59B6'),
}

for label, (value, color) in percentiles.items():
    fig_hist.add_vline(
        x=value,
        line_dash="dash",
        line_color=color,
        annotation_text=f"{label}: {value:.1f}ms",
        annotation_position="top"
    )

fig_hist.update_layout(
    title=f"Latency Distribution (n={len(df):,} samples)",
    xaxis_title="Latency (ms)",
    yaxis_title="Frequency",
    bargap=0.1,
    height=600,
    template="plotly_white",
    hovermode='x unified'
)

fig_hist.show()

In [231]:
# Create interactive CDF with Plotly
# Sort values for CDF
sorted_latencies = np.sort(df['latency_ms'])
cdf_values = np.arange(1, len(sorted_latencies) + 1) / len(sorted_latencies)

fig_cdf = go.Figure()

# Add CDF line
fig_cdf.add_trace(go.Scatter(
    x=sorted_latencies,
    y=cdf_values,
    mode='lines',
    name='CDF',
    line=dict(color='#34495E', width=3),
    hovertemplate='Latency: %{x:.1f}ms<br>Percentile: %{y:.1%}<extra></extra>'
))

# Add markers for key percentiles
key_percentiles = [
    (0.50, stats['p50'], 'P50', '#E74C3C'),
    (0.90, stats['p90'], 'P90', '#F39C12'),
    (0.95, stats['p95'], 'P95', '#3498DB'),
    (0.99, stats['p99'], 'P99', '#2ECC71'),
    (0.999, stats['p999'], 'P99.9', '#9B59B6'),
]

for percentile, value, label, color in key_percentiles:
    fig_cdf.add_trace(go.Scatter(
        x=[value],
        y=[percentile],
        mode='markers+text',
        name=label,
        marker=dict(size=12, color=color),
        text=[f"{label}: {value:.1f}ms"],
        textposition="top right",
        textfont=dict(size=10),
        showlegend=True
    ))
    
    # Add horizontal line
    fig_cdf.add_shape(
        type="line",
        x0=0, x1=value,
        y0=percentile, y1=percentile,
        line=dict(color=color, width=1, dash="dot"),
    )
    
    # Add vertical line
    fig_cdf.add_shape(
        type="line",
        x0=value, x1=value,
        y0=0, y1=percentile,
        line=dict(color=color, width=1, dash="dot"),
    )

fig_cdf.update_layout(
    title=f"Cumulative Distribution Function (CDF) - {len(df):,} samples",
    xaxis_title="Latency (ms)",
    yaxis_title="Percentile",
    yaxis_tickformat=".1%",
    height=700,
    template="plotly_white",
    hovermode='x unified',
    legend=dict(
        yanchor="bottom",
        y=0.5,
        xanchor="right",
        x=0.99
    )
)

# Add grid
fig_cdf.update_xaxes(showgrid=True, gridwidth=1, gridcolor='LightGray')
fig_cdf.update_yaxes(showgrid=True, gridwidth=1, gridcolor='LightGray')

fig_cdf.show()

In [232]:
# Create box plot by node to see distribution across nodes
fig_box = px.box(
    df, 
    x='node', 
    y='latency_ms',
    title='Latency Distribution by Node',
    labels={'latency_ms': 'Latency (ms)', 'node': 'Node'},
    height=500
)

fig_box.update_layout(
    template="plotly_white",
    xaxis_tickangle=-45
)

fig_box.show()